In [9]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')

from dotenv import load_dotenv
import os
load_dotenv(PROJECT_ROOT / ".env")

api_key = os.getenv("OPENAI_API_KEY")


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\semlu\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\semlu\AppData\Roaming\nltk_data...
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\semlu\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [40]:
!pip install sentence-transformers

  Using cached sentence_transformers-5.5.1-py3-none-any.whl.metadata (18 kB)
  Using cached transformers-5.9.0-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.17.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
Using cached sentence_transformers-5.5.1-py3-none-any.whl (588 kB)
Using cached huggingface_hub-1.17.0-py3-none-any.whl (671 kB)
Using cached transformers-5.9.0-py3-none-any.whl (10.8 MB)
Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl (2.7 MB)


In [41]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer("all-MiniLM-L6-v2")

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   --------------------------------------- 588.9/588.9 kB 10.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/671.5 kB ? eta -:--:--
   --------------------------------------- 671.5/671.5 kB 13.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/123.0 MB ? eta -:--:--
   - -------------------------------------- 6.0/123.0 MB 28.4 MB/s eta 0:00:05
   ---- ----------------------------------- 13.4/123.0 MB 32.3 MB/s eta 0:00:04
   ------ --------------------------------- 21.5/123.0 MB 34.0 MB/s eta 0:00:03
   --------- ------------------------------ 29.4/123.0 MB 34.5 MB/s eta 0:00:03
   ------------ --------------------------- 37.7/123.0 MB 35.8 MB/s eta 0:00:03
   --------------- ------------------------ 46.4/123.0 MB 36.4 MB/s eta 0:00:03
   ----------------- ---------------------- 55.1/123.0 MB 36.9 MB/s eta 0:00:02
   ---

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
wsproto 1.3.2 requires h11<1,>=0.16.0, but you have h11 0.14.0 which is incompatible.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\semlu\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\semlu\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [78]:
!pip install openai

   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 23.5 MB/s eta 0:00:00


In [79]:
from openai import OpenAI

client = OpenAI()

In [4]:
vse_df = pd.read_csv(PROCESSED_DIR / "vse_syllabi_processed.csv")
partner_df = pd.read_csv(PROCESSED_DIR / "partner_syllabi_processed.csv")

print("VŠE dataset:", vse_df.shape)
print("Partnerský dataset:", partner_df.shape)

display(vse_df.head(2))
display(partner_df.head(2))

VŠE dataset: (103, 17)
Partnerský dataset: (898, 17)


,course_title,course_code,university,country,source_url,academic_year,semester,semester_normalized,study_level,credits,credit_system,ects_credits,aims,course_content,learning_outcomes,assessment_methods,literature
0,Economics,2OP811,Prague University of Economics and Business,Czech Republic,https://insis-test.vse.cz/katalog/syllabus.pl?...,2026/2027,WS 2026/2027,winter,bachelor,6.0,ECTS,6.0,The couse is focused on basic economic categor...,1.Introduction into economics (allowance 1/0) ...,- identify the scope of economic research - de...,Term paper: 40 % Final test: 60 %,"Basic: Holman, R. (2005). Ekonomie. C.H. Beck...."
1,Financial Accounting,2OP816,Prague University of Economics and Business,Czech Republic,https://insis-test.vse.cz/katalog/syllabus.pl?...,2026/2027,WS 2026/2027,winter,bachelor,6.0,ECTS,6.0,Accounting as an information tool for the deci...,"1.Introduction to accounting: the nature, prin...",- understand the content of financial statemen...,Active lecture/seminar/workshop/tutorial parti...,"Basic: Janhuba, M., Míková, M., Roubíčková, J...."


,course_title,course_code,university,country,source_url,academic_year,semester,semester_normalized,study_level,credits,credit_system,ects_credits,aims,course_content,learning_outcomes,assessment_methods,literature
0,International Business,BST 2413,BI Norwegian Business School,Norway,https://programmeinfo.bi.no/nb/course/BST-2413...,2024/2025,Autumn,winter,bachelor,15.0,ECTS,15.0,This course prepares students for working in a...,The course content will be structured as below...,During the course students shall: Acquire know...,Exam category: Submission Form of assessment: ...,NaN
1,Mathematics for Data Science,EBA 1180,BI Norwegian Business School,Norway,https://programmeinfo.bi.no/nb/course/EBA-1180...,2024/2025,Autumn,winter,bachelor,7.5,ECTS,7.5,This is a basic course in mathematics. The cou...,Elementary algebra Mathematics of finance and ...,will have; Acquired a broad understanding of c...,Exam category: School Exam Form of assessment:...,NaN


In [5]:
print(vse_df.columns.tolist())

['course_title', 'course_code', 'university', 'country', 'source_url', 'academic_year', 'semester', 'semester_normalized', 'study_level', 'credits', 'credit_system', 'ects_credits', 'aims', 'course_content', 'learning_outcomes', 'assessment_methods', 'literature']


In [6]:
ATTRIBUTE = "course_content"

print("VŠE:")
print(vse_df[ATTRIBUTE].notna().sum())

print("\nPartneři:")
print(partner_df[ATTRIBUTE].notna().sum())

VŠE:
102

Partneři:
254


In [7]:
vse_df[ATTRIBUTE].dropna().head(3)

0    1.Introduction into economics (allowance 1/0) ...
1    1.Introduction to accounting: the nature, prin...
2    1.Marketing and commercial communication (allo...
Name: course_content, dtype: object

In [10]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_for_tfidf(text):
    if pd.isna(text) or text == '':
        return ''

    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)

    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]

    return ' '.join(tokens)

In [11]:
sample_text = partner_df["aims"].dropna().iloc[0]

print("PŮVODNÍ TEXT:")
print(sample_text[:700])

print("\nPREPROCESSED TEXT:")
print(preprocess_for_tfidf(sample_text)[:700])

PŮVODNÍ TEXT:
This course prepares students for working in an international business environment, by addressing critical macro-level factors in the environment and meso-level factors in the organization that are fundamental for a manager to be competent in international business. This course will provide students with knowledge of the institutional, organizational, and interpersonal factors and challenges that are critical to successful international business. The course begins with an overview of the global business environment and introduces the international organization and key decisions managers have to face when going international. The environmental and organizational context will provide the backd

PREPROCESSED TEXT:
course prepares student working international business environment addressing critical macro level factor environment meso level factor organization fundamental manager competent international business course provide student knowledge institutional organizational i

In [12]:
ATTRIBUTE = "aims"

partner_texts = partner_df[ATTRIBUTE].apply(preprocess_for_tfidf)
vse_texts = vse_df[ATTRIBUTE].apply(preprocess_for_tfidf)

print("Počet partnerských textů:", len(partner_texts))
print("Neprázdné partnerské texty:", (partner_texts != "").sum())

print("Počet VŠE textů:", len(vse_texts))
print("Neprázdné VŠE texty:", (vse_texts != "").sum())

Počet partnerských textů: 898
Neprázdné partnerské texty: 897
Počet VŠE textů: 103
Neprázdné VŠE texty: 103


In [14]:
vse_df[
    vse_df["course_code"].str.startswith("4IT", na=False)
][["course_code", "course_title"]].sort_values("course_code")

,course_code,course_title
48,4IT419,Architectures of data-oriented solutions
50,4IT526,Data warehousing and reporting
49,4IT529,Competitive Intelligence
52,4IT533,Advanced machine learning and big data
47,4IT534,Applied deep learning and artificial intelligence
51,4IT535,Machine learning operations
55,4IT537,Management of Data and Analytics for Business
54,4IT543,Presentation and communication skills
53,4IT553,Practice in applied data analytics


In [15]:
VSE_CODE = "4IT537"

vse_course = vse_df.loc[
    vse_df["course_code"] == VSE_CODE
].iloc[0]

print("Kód:", vse_course["course_code"])
print("Název:", vse_course["course_title"])

vse_text = preprocess_for_tfidf(vse_course["aims"])

print("\nTF-IDF text:")
print(vse_text[:1000])

Kód: 4IT537
Název: Management of Data and Analytics for Business

TF-IDF text:
course focus managing data analytics strictly aligned business need cover method alignment business goal management service aimed data utilization management project data analytics field management financial organizational aspect data analytics application finally data governance data management discussed necessary requirement sustainable practice using data support business organization


In [16]:
vectorizer = TfidfVectorizer()

partner_matrix = vectorizer.fit_transform(partner_texts)
vse_vector = vectorizer.transform([vse_text])

scores = cosine_similarity(vse_vector, partner_matrix).flatten()

print("Počet výsledků:", len(scores))
print("Min score:", scores.min())
print("Max score:", scores.max())

Počet výsledků: 898
Min score: 0.0
Max score: 0.533748823302367


In [17]:
results = partner_df.copy()

results["similarity_score"] = scores

results = results.sort_values(
    "similarity_score",
    ascending=False
)

results[
    [
        "course_code",
        "course_title",
        "university",
        "similarity_score"
    ]
].head(20)

,course_code,course_title,university,similarity_score
878,COMM 392,Introduction to Data Management and Analytics ...,Queen's University,0.533749
631,DASC2210,A Survey on Big Data in Science and Society,The Hong Kong University of Science and Techno...,0.481339
607,RMBI1020,Business Intelligence for Data-Driven Decisions,The Hong Kong University of Science and Techno...,0.415380
214,INFO260,Data Management,University of Canterbury,0.406229
680,COMP4332,Big Data Mining and Management,The Hong Kong University of Science and Techno...,0.404528
427,ACCT4710,Accounting Analytics for Professional Accountants,The Hong Kong University of Science and Techno...,0.381548
507,MARK3620,Marketing Analytics,The Hong Kong University of Science and Techno...,0.352663
614,RMBI4310,Advanced Data Mining for Risk Management and B...,The Hong Kong University of Science and Techno...,0.344659
592,ISOM4860,Decision and Data Analytics in Financial Markets,The Hong Kong University of Science and Techno...,0.339254
115,TEM 0056,Applied Data Analytics,BI Norwegian Business School,0.327170


In [18]:
top3 = results.head(3)

for _, row in top3.iterrows():
    print("=" * 120)
    print(row["course_title"])
    print(row["university"])
    print(f"Score: {row['similarity_score']:.3f}")
    print()
    print(row["aims"][:1500])
    print()

Introduction to Data Management and Analytics for Business
Queen's University
Score: 0.534

This course provides an introduction to data management and analytics for business. The course will present a systematic view of analytics with a key focus on using data for business intelligence and descriptive analytics as well as an overview of predictive analytics and how they are used by businesses for creating a competitive advantage. The course emphasizes the managerial aspects of analytics along with applications and implementation challenges, rather than technical issues (e.g. coding). The course will also focus on how to understand and manage data as an asset that is at the core of value creation in today's digital age, with emphasis on technical aspects such as data structure, data models, and data preparation and exploration, as well as management aspects such as data reliability and quality. Students will learn and apply relevant business intelligence and analytics concepts for solv

In [19]:
print(vse_course["aims"])

The course focuses on managing data and analytics that is strictly aligned with business needs. It covers methods of alignment with business goals, management of services aimed at data utilization, management of projects in data and analytics field and management of financial and organizational aspects of data and analytics applications. Finally, Data Governance and Data Management are discussed as necessary requirements of the sustainable practice of using data to support business in an organization.


In [20]:
ATTRIBUTE = "course_content"

partner_texts = partner_df[ATTRIBUTE].apply(preprocess_for_tfidf)
vse_text = preprocess_for_tfidf(vse_course[ATTRIBUTE])

print("Neprázdné partnerské texty:", (partner_texts != "").sum())
print("VŠE text je prázdný:", vse_text == "")

print("\nVŠE course_content:")
print(vse_course[ATTRIBUTE][:1500])

Neprázdné partnerské texty: 254
VŠE text je prázdný: False

VŠE course_content:
Strategy and business-IT alignment - Strategy of data and analytics - contents and relationships to other components of an organization Services management - Design and implementation of data and analytics services - Evaluation of quality of services - Sourcing (sourcing strategy, tenders, RFPs) - Operations of the business analytics environment - Technological resources - analysis, evaluation, configuration Organizational aspects - Management of human resources for business analytics - Management of relationships with external partners - Organizational aspects impacting Agile architectures and Cloud / hybrid environment applicability Data Governance & Data Management - Data Governance principles, areas of Data Management - Data Management goals definition concerning organizational goals and strategy - Roles in Data Management, data ownership, identification of business requirements on data and their qualit

In [21]:
vectorizer = TfidfVectorizer()

partner_matrix = vectorizer.fit_transform(partner_texts)
vse_vector = vectorizer.transform([vse_text])

scores = cosine_similarity(vse_vector, partner_matrix).flatten()

results = partner_df.copy()
results["similarity_score"] = scores

results = results.sort_values(
    "similarity_score",
    ascending=False
)

results[
    [
        "course_code",
        "course_title",
        "university",
        "similarity_score"
    ]
].head(10)

,course_code,course_title,university,similarity_score
5,EBA 3630,Data Driven Management Accounting,BI Norwegian Business School,0.444575
315,BUST08032,Business Analytics and Information Systems,University of Edinburgh,0.398214
2,EBA 3400,"Programming, Data Extraction and Visualisation",BI Norwegian Business School,0.383450
58,GRA 4137,Data Protection and Ethics in the Modern Busin...,BI Norwegian Business School,0.354161
12,ELE 3709,Project Management,BI Norwegian Business School,0.308649
826,IIO-14180,Project Management and Evaluation,Instituto Tecnológico Autónomo de México,0.287752
96,TEM 0241,AI for Business,BI Norwegian Business School,0.263604
8,EDI 3500,Low Code Software Development,BI Norwegian Business School,0.254929
365,BUST10145,Predictive Analytics for Business,University of Edinburgh,0.229368
318,BUST08033,Business Research Methods I: Introduction to D...,University of Edinburgh,0.225663


In [22]:
top5 = results.head(5)

for _, row in top5.iterrows():
    print("=" * 120)
    print(row["course_title"])
    print(row["university"])
    print(f"Score: {row['similarity_score']:.3f}")
    print()
    print(row["course_content"][:2000])
    print()

Data Driven Management Accounting
BI Norwegian Business School
Score: 0.445

In-house data External relevant data Extraction of data Decision analysis Management science Cost estimation for various profitability calculations Customer profitability analysis Accounting forensics Predictive analytics Visualisation of management accounting analytics

Business Analytics and Information Systems
University of Edinburgh
Score: 0.398

This course discusses how business analytics and information system tools can be used in synergy to address a variety of business problems. As for the information system part, the emphasis is placed on 1. the illustration of how data can be modelled, stored and retrieved, 2. the application of data management tools to tackle business problems. The focus of business analytics is on: 1. the introduction of a range of prescriptive analytics tools which has been shown to aid decision-making in practice, 2. the utilisation of software with appropriate data to formulate

In [23]:
ATTRIBUTE = "learning_outcomes"

partner_texts = partner_df[ATTRIBUTE].apply(preprocess_for_tfidf)
vse_text = preprocess_for_tfidf(vse_course[ATTRIBUTE])

print("Neprázdné partnerské texty:", (partner_texts != "").sum())
print("VŠE text je prázdný:", vse_text == "")

print("\nVŠE learning_outcomes:")
print(vse_course[ATTRIBUTE])

Neprázdné partnerské texty: 734
VŠE text je prázdný: False

VŠE learning_outcomes:
understand and discuss: - Necessities of management of data and analytics alignment with goals of an organization and how to achieve it - Approaches to managing data-oriented services - Which organizational aspects are crucial concerning data management, what are the necessary roles and responsibilities that need to be established in an organization and for what purposes - How metadata supports the sustainable development of data environment in an organization - What are the specifics and options in the management of data-oriented projects


In [24]:
vectorizer = TfidfVectorizer()

partner_matrix = vectorizer.fit_transform(partner_texts)
vse_vector = vectorizer.transform([vse_text])

scores = cosine_similarity(vse_vector, partner_matrix).flatten()

results = partner_df.copy()
results["similarity_score"] = scores

results = results.sort_values(
    "similarity_score",
    ascending=False
)

results[
    [
        "course_code",
        "course_title",
        "university",
        "similarity_score"
    ]
].head(10)

,course_code,course_title,university,similarity_score
631,DASC2210,A Survey on Big Data in Science and Society,The Hong Kong University of Science and Techno...,0.263237
58,GRA 4137,Data Protection and Ethics in the Modern Busin...,BI Norwegian Business School,0.227079
680,COMP4332,Big Data Mining and Management,The Hong Kong University of Science and Techno...,0.217071
215,INFO261,Introduction to Business Analytics,University of Canterbury,0.212698
639,DASC4300,Capstone Project for Data Analytics in Science,The Hong Kong University of Science and Techno...,0.210659
592,ISOM4860,Decision and Data Analytics in Financial Markets,The Hong Kong University of Science and Techno...,0.207683
561,ISOM3390,Business Programming in R,The Hong Kong University of Science and Techno...,0.204524
107,TEM 0203,Advanced Topics in Organisation Science,BI Norwegian Business School,0.195187
633,DASC3120,Data Structures for Data Analytics in Science,The Hong Kong University of Science and Techno...,0.192721
492,FINA4713,Introduction to Artificial Intelligence and Bi...,The Hong Kong University of Science and Techno...,0.186505


In [25]:
top5 = results.head(5)

for _, row in top5.iterrows():
    print("=" * 120)
    print(row["course_title"])
    print(row["university"])
    print(f"Score: {row['similarity_score']:.3f}")
    print()
    print(row["learning_outcomes"])
    print()

A Survey on Big Data in Science and Society
The Hong Kong University of Science and Technology
Score: 0.263

1. Describe some examples in the field of Data Science. 2. Describe the data analytics workflow. 3. Assess how data science affects various aspects of science and society. 4. Apply suitable data analytics tools and algorithms to solve problems.

Data Protection and Ethics in the Modern Business Environment
BI Norwegian Business School
Score: 0.227

This course will provide a comprehensive understanding of the legal and regulatory frameworks governing data protection and privacy. Students will learn about the essential requirements of data protection law in the European Union (EU) and the European Economic Area (EEA) , as well as the ethical implications and competitive advantages of using personal information in doing business. Throughout the course, students will: Learn about key terminology related to data protection Gain a basic understanding of the rules and principles for p

In [28]:
def tfidf_similarity(vse_text, partner_texts):
    vectorizer = TfidfVectorizer()

    partner_matrix = vectorizer.fit_transform(partner_texts)
    vse_vector = vectorizer.transform([vse_text])

    return cosine_similarity(vse_vector, partner_matrix).flatten()

In [27]:
WEIGHTS = {
    "aims": 0.40,
    "course_content": 0.35,
    "learning_outcomes": 0.25
}

In [29]:
aims_scores = tfidf_similarity(
    preprocess_for_tfidf(vse_course["aims"]),
    partner_df["aims"].apply(preprocess_for_tfidf)
)

content_scores = tfidf_similarity(
    preprocess_for_tfidf(vse_course["course_content"]),
    partner_df["course_content"].apply(preprocess_for_tfidf)
)

outcomes_scores = tfidf_similarity(
    preprocess_for_tfidf(vse_course["learning_outcomes"]),
    partner_df["learning_outcomes"].apply(preprocess_for_tfidf)
)

In [30]:
results = partner_df.copy()

results["aims_score"] = aims_scores
results["content_score"] = content_scores
results["outcomes_score"] = outcomes_scores

In [31]:
results["final_score_raw"] = (
    results["aims_score"] * WEIGHTS["aims"]
    + results["content_score"] * WEIGHTS["course_content"]
    + results["outcomes_score"] * WEIGHTS["learning_outcomes"]
)

In [34]:
def has_text(value):
    return not pd.isna(value) and str(value).strip() != ""

results["available_weight"] = 0.0

for attribute, weight in WEIGHTS.items():
    results["available_weight"] += results[attribute].apply(
        lambda x: weight if has_text(x) else 0
    )

In [35]:
results["weighted_sum"] = (
    results["aims_score"] * WEIGHTS["aims"] * results["aims"].apply(has_text)
    + results["content_score"] * WEIGHTS["course_content"] * results["course_content"].apply(has_text)
    + results["outcomes_score"] * WEIGHTS["learning_outcomes"] * results["learning_outcomes"].apply(has_text)
)

results["final_score"] = results["weighted_sum"] / results["available_weight"]

In [36]:
results.sort_values(
    "final_score",
    ascending=False
)[
    [
        "course_code",
        "course_title",
        "university",
        "aims_score",
        "content_score",
        "outcomes_score",
        "available_weight",
        "final_score"
    ]
].head(15)

,course_code,course_title,university,aims_score,content_score,outcomes_score,available_weight,final_score
878,COMM 392,Introduction to Data Management and Analytics ...,Queen's University,0.533749,0.000000,0.000000,0.40,0.533749
214,INFO260,Data Management,University of Canterbury,0.406229,0.000000,0.000000,0.40,0.406229
631,DASC2210,A Survey on Big Data in Science and Society,The Hong Kong University of Science and Techno...,0.481339,0.000000,0.263237,0.65,0.397454
680,COMP4332,Big Data Mining and Management,The Hong Kong University of Science and Techno...,0.404528,0.000000,0.217071,0.65,0.332429
427,ACCT4710,Accounting Analytics for Professional Accountants,The Hong Kong University of Science and Techno...,0.381548,0.000000,0.180872,0.65,0.304365
607,RMBI1020,Business Intelligence for Data-Driven Decisions,The Hong Kong University of Science and Techno...,0.415380,0.000000,0.091394,0.65,0.290770
5,EBA 3630,Data Driven Management Accounting,BI Norwegian Business School,0.229233,0.444575,0.168307,1.00,0.289371
592,ISOM4860,Decision and Data Analytics in Financial Markets,The Hong Kong University of Science and Techno...,0.339254,0.000000,0.207683,0.65,0.288650
166,DATA303,Computational Data Methods,University of Canterbury,0.273608,0.000000,0.000000,0.40,0.273608
614,RMBI4310,Advanced Data Mining for Risk Management and B...,The Hong Kong University of Science and Techno...,0.344659,0.000000,0.159284,0.65,0.273361


In [37]:
results[
    [
        "course_title",
        "aims",
        "course_content",
        "learning_outcomes"
    ]
].iloc[878]

course_title         Introduction to Data Management and Analytics ...
aims                 This course provides an introduction to data m...
course_content                                                     NaN
learning_outcomes                                                  NaN
Name: 878, dtype: object

In [42]:
def minimal_clean_text(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""

    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [43]:
sample_text = partner_df["aims"].dropna().iloc[0]

print("PŮVODNÍ TEXT:")
print(sample_text[:700])

print("\nMINIMAL CLEAN TEXT:")
print(minimal_clean_text(sample_text)[:700])

PŮVODNÍ TEXT:
This course prepares students for working in an international business environment, by addressing critical macro-level factors in the environment and meso-level factors in the organization that are fundamental for a manager to be competent in international business. This course will provide students with knowledge of the institutional, organizational, and interpersonal factors and challenges that are critical to successful international business. The course begins with an overview of the global business environment and introduces the international organization and key decisions managers have to face when going international. The environmental and organizational context will provide the backd

MINIMAL CLEAN TEXT:
This course prepares students for working in an international business environment, by addressing critical macro-level factors in the environment and meso-level factors in the organization that are fundamental for a manager to be competent in international busines

In [44]:
ATTRIBUTE = "aims"

partner_texts_sbert = partner_df[ATTRIBUTE].apply(minimal_clean_text).tolist()
vse_text_sbert = minimal_clean_text(vse_course[ATTRIBUTE])

partner_embeddings = sbert_model.encode(
    partner_texts_sbert,
    convert_to_numpy=True,
    show_progress_bar=True
)

vse_embedding = sbert_model.encode(
    [vse_text_sbert],
    convert_to_numpy=True
)

sbert_scores = cosine_similarity(
    vse_embedding,
    partner_embeddings
).flatten()

print("Počet výsledků:", len(sbert_scores))
print("Min score:", sbert_scores.min())
print("Max score:", sbert_scores.max())

Batches:   0%|          | 0/29 [00:00<?, ?it/s]

Počet výsledků: 898
Min score: -0.053975258
Max score: 0.7311598


In [45]:
sbert_results = partner_df.copy()
sbert_results["sbert_aims_score"] = sbert_scores

sbert_results = sbert_results.sort_values(
    "sbert_aims_score",
    ascending=False
)

sbert_results[
    [
        "course_code",
        "course_title",
        "university",
        "sbert_aims_score"
    ]
].head(15)

,course_code,course_title,university,sbert_aims_score
878,COMM 392,Introduction to Data Management and Analytics ...,Queen's University,0.731160
895,COMM 461,Data Science for Business,Queen's University,0.721355
215,INFO261,Introduction to Business Analytics,University of Canterbury,0.716649
427,ACCT4710,Accounting Analytics for Professional Accountants,The Hong Kong University of Science and Techno...,0.703007
214,INFO260,Data Management,University of Canterbury,0.701276
167,DATA309,Data Science Capstone Project,University of Canterbury,0.672651
115,TEM 0056,Applied Data Analytics,BI Norwegian Business School,0.669124
607,RMBI1020,Business Intelligence for Data-Driven Decisions,The Hong Kong University of Science and Techno...,0.665496
638,DASC4020,Structured Query Language for Data Analytics,The Hong Kong University of Science and Techno...,0.662415
798,ADM-12350,Data-driven business decisions,Instituto Tecnológico Autónomo de México,0.639495


In [46]:
top5 = sbert_results.head(5)

for _, row in top5.iterrows():
    print("=" * 120)
    print(row["course_title"])
    print(row["university"])
    print(f"Score: {row['sbert_aims_score']:.3f}")
    print()
    print(row["aims"])
    print()

Introduction to Data Management and Analytics for Business
Queen's University
Score: 0.731

This course provides an introduction to data management and analytics for business. The course will present a systematic view of analytics with a key focus on using data for business intelligence and descriptive analytics as well as an overview of predictive analytics and how they are used by businesses for creating a competitive advantage. The course emphasizes the managerial aspects of analytics along with applications and implementation challenges, rather than technical issues (e.g. coding). The course will also focus on how to understand and manage data as an asset that is at the core of value creation in today's digital age, with emphasis on technical aspects such as data structure, data models, and data preparation and exploration, as well as management aspects such as data reliability and quality. Students will learn and apply relevant business intelligence and analytics concepts for solv

In [47]:
ATTRIBUTE = "course_content"

partner_texts_sbert = (
    partner_df[ATTRIBUTE]
    .fillna("")
    .apply(minimal_clean_text)
    .tolist()
)

vse_text_sbert = minimal_clean_text(
    vse_course[ATTRIBUTE]
)

print("Neprázdných partnerů:", sum(t != "" for t in partner_texts_sbert))
print("VŠE text prázdný:", vse_text_sbert == "")

Neprázdných partnerů: 254
VŠE text prázdný: False


In [48]:
partner_embeddings = sbert_model.encode(
    partner_texts_sbert,
    convert_to_numpy=True,
    show_progress_bar=True
)

vse_embedding = sbert_model.encode(
    [vse_text_sbert],
    convert_to_numpy=True
)

sbert_scores = cosine_similarity(
    vse_embedding,
    partner_embeddings
).flatten()

Batches:   0%|          | 0/29 [00:00<?, ?it/s]

In [49]:
sbert_results = partner_df.copy()

sbert_results["sbert_content_score"] = sbert_scores

sbert_results = sbert_results.sort_values(
    "sbert_content_score",
    ascending=False
)

sbert_results[
    [
        "course_code",
        "course_title",
        "university",
        "sbert_content_score"
    ]
].head(15)

,course_code,course_title,university,sbert_content_score
315,BUST08032,Business Analytics and Information Systems,University of Edinburgh,0.643415
5,EBA 3630,Data Driven Management Accounting,BI Norwegian Business School,0.559763
8,EDI 3500,Low Code Software Development,BI Norwegian Business School,0.525373
9,EDI 3510,Business Information Systems,BI Norwegian Business School,0.516011
317,BUST08056,Business Research Methods 2: Research Design a...,University of Edinburgh,0.493451
318,BUST08033,Business Research Methods I: Introduction to D...,University of Edinburgh,0.484180
12,ELE 3709,Project Management,BI Norwegian Business School,0.455190
93,TEM 0480,Operational Planning and Logistics Analytics,BI Norwegian Business School,0.424383
116,tem 0400,Business Economics,BI Norwegian Business School,0.420266
373,BUST10159,The Strategy of Digital Transformation,University of Edinburgh,0.419752


In [50]:
ATTRIBUTE = "learning_outcomes"

partner_texts_sbert = (
    partner_df[ATTRIBUTE]
    .fillna("")
    .apply(minimal_clean_text)
    .tolist()
)

vse_text_sbert = minimal_clean_text(
    vse_course[ATTRIBUTE]
)

print("Neprázdných partnerů:", sum(t != "" for t in partner_texts_sbert))
print("VŠE text prázdný:", vse_text_sbert == "")

Neprázdných partnerů: 734
VŠE text prázdný: False


In [51]:
partner_embeddings = sbert_model.encode(
    partner_texts_sbert,
    convert_to_numpy=True,
    show_progress_bar=True
)

vse_embedding = sbert_model.encode(
    [vse_text_sbert],
    convert_to_numpy=True
)

sbert_scores = cosine_similarity(
    vse_embedding,
    partner_embeddings
).flatten()

Batches:   0%|          | 0/29 [00:00<?, ?it/s]

In [52]:
sbert_results = partner_df.copy()

sbert_results["sbert_outcomes_score"] = sbert_scores

sbert_results = sbert_results.sort_values(
    "sbert_outcomes_score",
    ascending=False
)

sbert_results[
    [
        "course_code",
        "course_title",
        "university",
        "sbert_outcomes_score"
    ]
].head(15)

,course_code,course_title,university,sbert_outcomes_score
215,INFO261,Introduction to Business Analytics,University of Canterbury,0.640514
633,DASC3120,Data Structures for Data Analytics in Science,The Hong Kong University of Science and Techno...,0.629594
631,DASC2210,A Survey on Big Data in Science and Society,The Hong Kong University of Science and Techno...,0.606101
551,ISOM3260,Database Design and Administration,The Hong Kong University of Science and Techno...,0.571939
208,INFO123,Business Information Systems and Technology,University of Canterbury,0.544508
630,DASC2110,Object-oriented Programming for Data Analytics...,The Hong Kong University of Science and Techno...,0.541616
638,DASC4020,Structured Query Language for Data Analytics,The Hong Kong University of Science and Techno...,0.535826
365,BUST10145,Predictive Analytics for Business,University of Edinburgh,0.534790
639,DASC4300,Capstone Project for Data Analytics in Science,The Hong Kong University of Science and Techno...,0.513318
666,COMP3311,Database Management Systems,The Hong Kong University of Science and Techno...,0.509831


In [53]:
top5 = sbert_results.head(5)

for _, row in top5.iterrows():
    print("=" * 120)
    print(row["course_title"])
    print(row["university"])
    print(f"Score: {row['sbert_outcomes_score']:.3f}")
    print()
    print(row["learning_outcomes"])
    print()

Introduction to Business Analytics
University of Canterbury
Score: 0.641

1) Demonstrate an understanding of key business data analytics concepts, tools and techniques. 2) To recognise and demonstrate an understanding of issues related to the use of business data for decision-making including data ownership and ethical considerations. 3) To recognise and analyse organisational problems and opportunities related to the role and use of data analytics in organisations. 4) To select and apply suitable techniques (e.g. regression, forecasting, cluster analysis, visualisation), extract meaningful insights from various data and make recommendations that align with the organisationsâ€™ context and objectives. 5) To perform key activities related to the analysis of organisational data and provide and communicate evidence-based recommendations.

Data Structures for Data Analytics in Science
The Hong Kong University of Science and Technology
Score: 0.630

1. Describe and implement various data st

In [54]:
partner_texts = partner_df["aims"].fillna("").apply(minimal_clean_text).tolist()
vse_text = minimal_clean_text(vse_course["aims"])

partner_embeddings = sbert_model.encode(
    partner_texts,
    convert_to_numpy=True
)

vse_embedding = sbert_model.encode(
    [vse_text],
    convert_to_numpy=True
)

aims_scores = cosine_similarity(
    vse_embedding,
    partner_embeddings
).flatten()

In [55]:
partner_texts = partner_df["course_content"].fillna("").apply(minimal_clean_text).tolist()
vse_text = minimal_clean_text(vse_course["course_content"])

partner_embeddings = sbert_model.encode(
    partner_texts,
    convert_to_numpy=True
)

vse_embedding = sbert_model.encode(
    [vse_text],
    convert_to_numpy=True
)

content_scores = cosine_similarity(
    vse_embedding,
    partner_embeddings
).flatten()

In [56]:
partner_texts = partner_df["learning_outcomes"].fillna("").apply(minimal_clean_text).tolist()
vse_text = minimal_clean_text(vse_course["learning_outcomes"])

partner_embeddings = sbert_model.encode(
    partner_texts,
    convert_to_numpy=True
)

vse_embedding = sbert_model.encode(
    [vse_text],
    convert_to_numpy=True
)

outcomes_scores = cosine_similarity(
    vse_embedding,
    partner_embeddings
).flatten()

In [57]:
sbert_results = partner_df.copy()

sbert_results["aims_score"] = aims_scores
sbert_results["content_score"] = content_scores
sbert_results["outcomes_score"] = outcomes_scores

In [58]:
def has_text(value):
    return not pd.isna(value) and str(value).strip() != ""

In [59]:
sbert_results["available_weight"] = 0.0

for attribute, weight in WEIGHTS.items():
    sbert_results["available_weight"] += sbert_results[attribute].apply(
        lambda x: weight if has_text(x) else 0
    )

In [60]:
sbert_results["weighted_sum"] = (
    sbert_results["aims_score"] * WEIGHTS["aims"] * sbert_results["aims"].apply(has_text)
    + sbert_results["content_score"] * WEIGHTS["course_content"] * sbert_results["course_content"].apply(has_text)
    + sbert_results["outcomes_score"] * WEIGHTS["learning_outcomes"] * sbert_results["learning_outcomes"].apply(has_text)
)

In [61]:
sbert_results["final_score"] = (
    sbert_results["weighted_sum"]
    / sbert_results["available_weight"]
)

In [62]:
sbert_results.sort_values(
    "final_score",
    ascending=False
)[
    [
        "course_code",
        "course_title",
        "university",
        "aims_score",
        "content_score",
        "outcomes_score",
        "available_weight",
        "final_score"
    ]
].head(20)

,course_code,course_title,university,aims_score,content_score,outcomes_score,available_weight,final_score
878,COMM 392,Introduction to Data Management and Analytics ...,Queen's University,0.731160,-0.013371,-0.049615,0.40,0.731160
895,COMM 461,Data Science for Business,Queen's University,0.721355,-0.013371,-0.049615,0.40,0.721355
214,INFO260,Data Management,University of Canterbury,0.701276,-0.013371,-0.049615,0.40,0.701276
215,INFO261,Introduction to Business Analytics,University of Canterbury,0.716649,-0.013371,0.640514,0.65,0.687366
167,DATA309,Data Science Capstone Project,University of Canterbury,0.672651,-0.013371,-0.049615,0.40,0.672651
427,ACCT4710,Accounting Analytics for Professional Accountants,The Hong Kong University of Science and Techno...,0.703007,-0.013371,0.500472,0.65,0.625109
638,DASC4020,Structured Query Language for Data Analytics,The Hong Kong University of Science and Techno...,0.662415,-0.013371,0.535826,0.65,0.613727
631,DASC2210,A Survey on Big Data in Science and Society,The Hong Kong University of Science and Techno...,0.593906,-0.013371,0.606101,0.65,0.598596
163,DATA201,Data Wrangling,University of Canterbury,0.593460,-0.013371,-0.049615,0.40,0.593460
607,RMBI1020,Business Intelligence for Data-Driven Decisions,The Hong Kong University of Science and Techno...,0.665496,-0.013371,0.461391,0.65,0.586994


In [63]:
def sbert_similarity_attribute(vse_text, partner_series):
    scores = np.full(len(partner_series), np.nan)

    valid_mask = (
        partner_series.notna()
        & (partner_series.astype(str).str.strip() != "")
    )

    valid_texts = (
        partner_series.loc[valid_mask]
        .apply(minimal_clean_text)
        .tolist()
    )

    partner_embeddings = sbert_model.encode(
        valid_texts,
        convert_to_numpy=True
    )

    vse_embedding = sbert_model.encode(
        [minimal_clean_text(vse_text)],
        convert_to_numpy=True
    )

    valid_scores = cosine_similarity(
        vse_embedding,
        partner_embeddings
    ).flatten()

    scores[valid_mask.to_numpy()] = valid_scores

    return scores

In [64]:
sbert_aims_scores = sbert_similarity_attribute(
    vse_course["aims"],
    partner_df["aims"]
)

sbert_content_scores = sbert_similarity_attribute(
    vse_course["course_content"],
    partner_df["course_content"]
)

sbert_outcomes_scores = sbert_similarity_attribute(
    vse_course["learning_outcomes"],
    partner_df["learning_outcomes"]
)

In [65]:
sbert_results = partner_df.copy()

sbert_results["aims_score"] = sbert_aims_scores
sbert_results["content_score"] = sbert_content_scores
sbert_results["outcomes_score"] = sbert_outcomes_scores

In [66]:
sbert_results["available_weight"] = 0.0

for attribute, weight in WEIGHTS.items():
    sbert_results["available_weight"] += (
        sbert_results[attribute]
        .notna()
        .astype(float)
        * weight
    )

In [67]:
sbert_results["weighted_sum"] = (
    sbert_results["aims_score"].fillna(0) * WEIGHTS["aims"]
    + sbert_results["content_score"].fillna(0) * WEIGHTS["course_content"]
    + sbert_results["outcomes_score"].fillna(0) * WEIGHTS["learning_outcomes"]
)

In [68]:
sbert_results["final_score"] = (
    sbert_results["weighted_sum"]
    / sbert_results["available_weight"]
)

In [69]:
sbert_results.sort_values(
    "final_score",
    ascending=False
)[
    [
        "course_code",
        "course_title",
        "university",
        "aims_score",
        "content_score",
        "outcomes_score",
        "available_weight",
        "final_score"
    ]
].head(20)

,course_code,course_title,university,aims_score,content_score,outcomes_score,available_weight,final_score
878,COMM 392,Introduction to Data Management and Analytics ...,Queen's University,0.731160,NaN,NaN,0.40,0.731160
895,COMM 461,Data Science for Business,Queen's University,0.721355,NaN,NaN,0.40,0.721355
214,INFO260,Data Management,University of Canterbury,0.701276,NaN,NaN,0.40,0.701276
215,INFO261,Introduction to Business Analytics,University of Canterbury,0.716649,NaN,0.640514,0.65,0.687366
167,DATA309,Data Science Capstone Project,University of Canterbury,0.672651,NaN,NaN,0.40,0.672651
427,ACCT4710,Accounting Analytics for Professional Accountants,The Hong Kong University of Science and Techno...,0.703007,NaN,0.500472,0.65,0.625109
638,DASC4020,Structured Query Language for Data Analytics,The Hong Kong University of Science and Techno...,0.662415,NaN,0.535826,0.65,0.613727
631,DASC2210,A Survey on Big Data in Science and Society,The Hong Kong University of Science and Techno...,0.593906,NaN,0.606101,0.65,0.598596
163,DATA201,Data Wrangling,University of Canterbury,0.593460,NaN,NaN,0.40,0.593460
607,RMBI1020,Business Intelligence for Data-Driven Decisions,The Hong Kong University of Science and Techno...,0.665496,NaN,0.461391,0.65,0.586994


In [70]:
partner_df[
    partner_df["learning_outcomes"]
    .str.contains("should be", case=False, na=False)
][["course_title", "learning_outcomes"]]

,course_title,learning_outcomes
0,International Business,During the course students shall: Acquire know...
7,Communication in Action: Dialogue and Discourse,Students will acquire advanced knowledge about...
8,Low Code Software Development,"After completing the course, the candidate wil..."
11,Digital Innovation,"After completing the course, a candidate will ..."
12,Project Management,During the course student shall: Obtain knowle...
13,"Financial Bubbles, Crashes and Crises","During the course, students shall acquire: Kno..."
18,Financial Technology,"The student will Understand the history, curre..."
21,"Global Sustainability: Climate, Environment an...",The aim of this course is to provide an unders...
28,Business Communication - Culture and Ethics,Intercultural awareness. Students will acquire...
29,Business Communication - Negotiations and Pres...,The overall objective of this course is that s...


In [74]:
partner_df["learning_outcomes"].str.contains(
    r"^students will",
    case=False,
    na=False
).sum()

8

In [75]:
partner_df["learning_outcomes"].str.contains(
    r"^will be able to",
    case=False,
    na=False
).sum()

2

In [76]:
from dotenv import load_dotenv
import os

load_dotenv(PROJECT_ROOT / ".env")

api_key = os.getenv("OPENAI_API_KEY")

print(api_key[:15])

sk-proj-sC3T0bX


In [80]:
response = client.embeddings.create(
    model="text-embedding-3-small",
    input="Data management and business analytics"
)

embedding = response.data[0].embedding

print(f"Délka embeddingu: {len(embedding)}")
print(embedding[:5])

Délka embeddingu: 1536
[0.00405120849609375, 0.038909912109375, 0.0882568359375, -0.017333984375, 0.023773193359375]


In [81]:
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

def get_openai_embedding(text):
    response = client.embeddings.create(
        model=OPENAI_EMBEDDING_MODEL,
        input=text
    )
    return response.data[0].embedding

In [82]:
ATTRIBUTE = "aims"

vse_text = vse_df.loc[
    vse_df["course_code"] == "4IT537",
    ATTRIBUTE
].iloc[0]

print(vse_text[:500])

The course focuses on managing data and analytics that is strictly aligned with business needs. It covers methods of alignment with business goals, management of services aimed at data utilization, management of projects in data and analytics field and management of financial and organizational aspects of data and analytics applications. Finally, Data Governance and Data Management are discussed as necessary requirements of the sustainable practice of using data to support business in an organiz


In [84]:
vse_embedding = get_openai_embedding(vse_text)

print(len(vse_embedding))

1536


In [85]:
ATTRIBUTE = "aims"

partner_texts = partner_df[ATTRIBUTE].dropna()

print("Počet partnerských textů:", len(partner_texts))

total_chars = partner_texts.str.len().sum()
avg_chars = partner_texts.str.len().mean()

print("Celkem znaků:", total_chars)
print("Průměr znaků:", round(avg_chars))

Počet partnerských textů: 897
Celkem znaků: 494449
Průměr znaků: 551


In [86]:
ATTRIBUTE = "aims"

partner_texts = partner_df[ATTRIBUTE].dropna()
partner_indices = partner_texts.index.tolist()

print(len(partner_texts))

897


In [87]:
def get_openai_embeddings(texts, batch_size=100):
    embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=batch
        )

        embeddings.extend(
            [item.embedding for item in response.data]
        )

        print(
            f"Hotovo {min(i + batch_size, len(texts))}/{len(texts)}"
        )

    return embeddings

In [88]:
partner_embeddings = get_openai_embeddings(
    partner_texts.tolist()
)

print(len(partner_embeddings))

Hotovo 100/897
Hotovo 200/897
Hotovo 300/897
Hotovo 400/897
Hotovo 500/897
Hotovo 600/897
Hotovo 700/897
Hotovo 800/897
Hotovo 897/897
897


In [89]:
partner_embeddings = np.array(partner_embeddings)

vse_embedding = np.array(get_openai_embedding(vse_text)).reshape(1, -1)

openai_scores = cosine_similarity(
    vse_embedding,
    partner_embeddings
).flatten()

print("Počet výsledků:", len(openai_scores))
print("Min score:", openai_scores.min())
print("Max score:", openai_scores.max())

Počet výsledků: 897
Min score: 0.12828219930314247
Max score: 0.7581550764666648


In [90]:
openai_results = partner_df.loc[partner_indices].copy()
openai_results["openai_aims_score"] = openai_scores

openai_results = openai_results.sort_values(
    "openai_aims_score",
    ascending=False
)

openai_results[
    [
        "course_code",
        "course_title",
        "university",
        "openai_aims_score"
    ]
].head(15)

,course_code,course_title,university,openai_aims_score
878,COMM 392,Introduction to Data Management and Analytics ...,Queen's University,0.758155
214,INFO260,Data Management,University of Canterbury,0.731719
895,COMM 461,Data Science for Business,Queen's University,0.698917
215,INFO261,Introduction to Business Analytics,University of Canterbury,0.672088
558,ISOM3360,Data Mining for Business Analytics,The Hong Kong University of Science and Techno...,0.663187
607,RMBI1020,Business Intelligence for Data-Driven Decisions,The Hong Kong University of Science and Techno...,0.655140
115,TEM 0056,Applied Data Analytics,BI Norwegian Business School,0.653757
680,COMP4332,Big Data Mining and Management,The Hong Kong University of Science and Techno...,0.647407
545,ISOM2600,Introduction to Business Analytics,The Hong Kong University of Science and Techno...,0.643160
220,INFO361,Business Intelligence and Analytics,University of Canterbury,0.641752


In [92]:
print(
    openai_results.loc[
        openai_results["course_code"] == "ELE 3781",
        "aims"
    ].iloc[0]
)

This elective course is based on the Master's course GRA 6035 Mathematics, and is intended for students at the programme Bachelor of Data Science for Business, aiming for admission to the Master of Data Science for Business. The language of mathematics is extensively used to analyse problems in business, economics and finance, and mathematical models, theories, and methods are extensively used to solve problems. The mathematical requirements of students in Business Analytics go beyond the material usually taught in other undergraduate courses, and this elective course will teach the student more advanced mathematical models, theories, and methods. Topics include linear algebra and matrix methods, optimisation in several real variables, differential and difference equations.


In [93]:
ATTRIBUTE = "course_content"

partner_texts = partner_df[ATTRIBUTE].dropna()
partner_indices = partner_texts.index.tolist()

print("Počet partnerských textů:", len(partner_texts))

total_chars = partner_texts.str.len().sum()
avg_chars = partner_texts.str.len().mean()

print("Celkem znaků:", total_chars)
print("Průměr znaků:", round(avg_chars))

Počet partnerských textů: 254
Celkem znaků: 269000
Průměr znaků: 1059


In [94]:
partner_embeddings = get_openai_embeddings(
    partner_texts.tolist()
)

print(len(partner_embeddings))

Hotovo 100/254
Hotovo 200/254
Hotovo 254/254
254


In [95]:
vse_text = vse_df.loc[
    vse_df["course_code"] == "4IT537",
    ATTRIBUTE
].iloc[0]

vse_embedding = get_openai_embedding(vse_text)

partner_embeddings = np.array(partner_embeddings)
vse_embedding = np.array(vse_embedding).reshape(1, -1)

openai_scores = cosine_similarity(
    vse_embedding,
    partner_embeddings
).flatten()

print("Počet výsledků:", len(openai_scores))
print("Min score:", openai_scores.min())
print("Max score:", openai_scores.max())

Počet výsledků: 254
Min score: 0.1994409685434121
Max score: 0.6323852294971442


In [96]:
openai_content_results = partner_df.loc[partner_indices].copy()
openai_content_results["openai_content_score"] = openai_scores

openai_content_results = openai_content_results.sort_values(
    "openai_content_score",
    ascending=False
)

openai_content_results[
    [
        "course_code",
        "course_title",
        "university",
        "openai_content_score"
    ]
].head(15)

,course_code,course_title,university,openai_content_score
315,BUST08032,Business Analytics and Information Systems,University of Edinburgh,0.632385
5,EBA 3630,Data Driven Management Accounting,BI Norwegian Business School,0.573533
373,BUST10159,The Strategy of Digital Transformation,University of Edinburgh,0.529203
329,BUST08057,Strategic Marketing Insights,University of Edinburgh,0.522947
368,CMSE10002,Strategic Management,University of Edinburgh,0.519675
363,BUST10092,Operations Strategy,University of Edinburgh,0.518740
37,EXC 3580,Marketing Management and Strategy,BI Norwegian Business School,0.518193
96,TEM 0241,AI for Business,BI Norwegian Business School,0.512872
9,EDI 3510,Business Information Systems,BI Norwegian Business School,0.508026
8,EDI 3500,Low Code Software Development,BI Norwegian Business School,0.507190


In [97]:
for code in ["BUST08032", "EBA 3630"]:
    row = openai_content_results[
        openai_content_results["course_code"] == code
    ].iloc[0]

    print(row["course_title"])
    print()
    print(row["course_content"])
    print("\n" + "="*100 + "\n")

Business Analytics and Information Systems

This course discusses how business analytics and information system tools can be used in synergy to address a variety of business problems. As for the information system part, the emphasis is placed on 1. the illustration of how data can be modelled, stored and retrieved, 2. the application of data management tools to tackle business problems. The focus of business analytics is on: 1. the introduction of a range of prescriptive analytics tools which has been shown to aid decision-making in practice, 2. the utilisation of software with appropriate data to formulate and execute various models, 3. the explanation of the findings from modelling a managerial situation to the relevant stakeholders. Outline Content L 1. Data Management and Database Design L 2. Entity-relationship L 3. Structured Query Language L 4. Introduction to Linear Programming L 5. Sensitivity Analysis and Advanced Applications in Linear Programming L 6. Group Project Proposal

In [98]:
ATTRIBUTE = "learning_outcomes"

partner_texts = partner_df[ATTRIBUTE].dropna()
partner_indices = partner_texts.index.tolist()

print("Počet partnerských textů:", len(partner_texts))

total_chars = partner_texts.str.len().sum()
avg_chars = partner_texts.str.len().mean()

print("Celkem znaků:", total_chars)
print("Průměr znaků:", round(avg_chars))

Počet partnerských textů: 734
Celkem znaků: 520679
Průměr znaků: 709


In [99]:
partner_embeddings = get_openai_embeddings(
    partner_texts.tolist()
)

print(len(partner_embeddings))

Hotovo 100/734
Hotovo 200/734
Hotovo 300/734
Hotovo 400/734
Hotovo 500/734
Hotovo 600/734
Hotovo 700/734
Hotovo 734/734
734


In [100]:
vse_text = vse_df.loc[
    vse_df["course_code"] == "4IT537",
    ATTRIBUTE
].iloc[0]

print(vse_text)

understand and discuss: - Necessities of management of data and analytics alignment with goals of an organization and how to achieve it - Approaches to managing data-oriented services - Which organizational aspects are crucial concerning data management, what are the necessary roles and responsibilities that need to be established in an organization and for what purposes - How metadata supports the sustainable development of data environment in an organization - What are the specifics and options in the management of data-oriented projects


In [101]:
vse_embedding = get_openai_embedding(vse_text)

print(len(vse_embedding))

1536


In [102]:
partner_embeddings = np.array(partner_embeddings)

vse_embedding = np.array(
    vse_embedding
).reshape(1, -1)

openai_scores = cosine_similarity(
    vse_embedding,
    partner_embeddings
).flatten()

print("Počet výsledků:", len(openai_scores))
print("Min score:", openai_scores.min())
print("Max score:", openai_scores.max())

Počet výsledků: 734
Min score: 0.15664111050480983
Max score: 0.636933527202246


In [103]:
openai_outcomes_results = partner_df.loc[
    partner_indices
].copy()

openai_outcomes_results[
    "openai_outcomes_score"
] = openai_scores

openai_outcomes_results = (
    openai_outcomes_results
    .sort_values(
        "openai_outcomes_score",
        ascending=False
    )
)

openai_outcomes_results[
    [
        "course_code",
        "course_title",
        "university",
        "openai_outcomes_score"
    ]
].head(15)

,course_code,course_title,university,openai_outcomes_score
215,INFO261,Introduction to Business Analytics,University of Canterbury,0.636934
614,RMBI4310,Advanced Data Mining for Risk Management and B...,The Hong Kong University of Science and Techno...,0.574518
365,BUST10145,Predictive Analytics for Business,University of Edinburgh,0.567909
639,DASC4300,Capstone Project for Data Analytics in Science,The Hong Kong University of Science and Techno...,0.556175
563,ISOM3530,Business Data Analytics,The Hong Kong University of Science and Techno...,0.550744
532,MGMT4290,HR Analytics,The Hong Kong University of Science and Techno...,0.549727
558,ISOM3360,Data Mining for Business Analytics,The Hong Kong University of Science and Techno...,0.549724
427,ACCT4710,Accounting Analytics for Professional Accountants,The Hong Kong University of Science and Techno...,0.548004
633,DASC3120,Data Structures for Data Analytics in Science,The Hong Kong University of Science and Techno...,0.531246
638,DASC4020,Structured Query Language for Data Analytics,The Hong Kong University of Science and Techno...,0.529579


In [104]:
aims_scores = openai_results[
    ["course_code", "openai_aims_score"]
]

content_scores = openai_content_results[
    ["course_code", "openai_content_score"]
]

outcomes_scores = openai_outcomes_results[
    ["course_code", "openai_outcomes_score"]
]

In [105]:
results = partner_df.copy()

results = results.merge(
    aims_scores,
    on="course_code",
    how="left"
)

results = results.merge(
    content_scores,
    on="course_code",
    how="left"
)

results = results.merge(
    outcomes_scores,
    on="course_code",
    how="left"
)

In [106]:
results["available_weight"] = 0.0

results["available_weight"] += (
    results["openai_aims_score"].notna().astype(float)
    * WEIGHTS["aims"]
)

results["available_weight"] += (
    results["openai_content_score"].notna().astype(float)
    * WEIGHTS["course_content"]
)

results["available_weight"] += (
    results["openai_outcomes_score"].notna().astype(float)
    * WEIGHTS["learning_outcomes"]
)

In [107]:
results["weighted_sum"] = (
    results["openai_aims_score"].fillna(0) * WEIGHTS["aims"]
    + results["openai_content_score"].fillna(0) * WEIGHTS["course_content"]
    + results["openai_outcomes_score"].fillna(0) * WEIGHTS["learning_outcomes"]
)

In [108]:
results["final_score"] = (
    results["weighted_sum"] / results["available_weight"]
)

In [109]:
results.sort_values(
    "final_score",
    ascending=False
)[
    [
        "course_code",
        "course_title",
        "university",
        "openai_aims_score",
        "openai_content_score",
        "openai_outcomes_score",
        "available_weight",
        "final_score"
    ]
].head(20)

,course_code,course_title,university,openai_aims_score,openai_content_score,openai_outcomes_score,available_weight,final_score
878,COMM 392,Introduction to Data Management and Analytics ...,Queen's University,0.758155,NaN,NaN,0.40,0.758155
214,INFO260,Data Management,University of Canterbury,0.731719,NaN,NaN,0.40,0.731719
895,COMM 461,Data Science for Business,Queen's University,0.698917,NaN,NaN,0.40,0.698917
215,INFO261,Introduction to Business Analytics,University of Canterbury,0.672088,NaN,0.636934,0.65,0.658567
220,INFO361,Business Intelligence and Analytics,University of Canterbury,0.641752,NaN,NaN,0.40,0.641752
558,ISOM3360,Data Mining for Business Analytics,The Hong Kong University of Science and Techno...,0.663187,NaN,0.549724,0.65,0.619547
427,ACCT4710,Accounting Analytics for Professional Accountants,The Hong Kong University of Science and Techno...,0.636578,NaN,0.548004,0.65,0.602511
680,COMP4332,Big Data Mining and Management,The Hong Kong University of Science and Techno...,0.647407,NaN,0.514817,0.65,0.596411
163,DATA201,Data Wrangling,University of Canterbury,0.590257,NaN,NaN,0.40,0.590257
545,ISOM2600,Introduction to Business Analytics,The Hong Kong University of Science and Techno...,0.643160,NaN,0.495821,0.65,0.586491


In [115]:
results.sort_values(
    "final_score",
    ascending=False
)[
    [
        "course_code",
        "course_title",
        "openai_aims_score",
        "openai_content_score",
        "openai_outcomes_score",
        "available_weight",
        "final_score"
    ]
].head(30)

,course_code,course_title,openai_aims_score,openai_content_score,openai_outcomes_score,available_weight,final_score
878,COMM 392,Introduction to Data Management and Analytics ...,0.758155,NaN,NaN,0.40,0.758155
214,INFO260,Data Management,0.731719,NaN,NaN,0.40,0.731719
895,COMM 461,Data Science for Business,0.698917,NaN,NaN,0.40,0.698917
215,INFO261,Introduction to Business Analytics,0.672088,NaN,0.636934,0.65,0.658567
220,INFO361,Business Intelligence and Analytics,0.641752,NaN,NaN,0.40,0.641752
558,ISOM3360,Data Mining for Business Analytics,0.663187,NaN,0.549724,0.65,0.619547
427,ACCT4710,Accounting Analytics for Professional Accountants,0.636578,NaN,0.548004,0.65,0.602511
680,COMP4332,Big Data Mining and Management,0.647407,NaN,0.514817,0.65,0.596411
163,DATA201,Data Wrangling,0.590257,NaN,NaN,0.40,0.590257
545,ISOM2600,Introduction to Business Analytics,0.643160,NaN,0.495821,0.65,0.586491


In [116]:
results_complete = results[
    (results["available_weight"] == 1.0)
]

results_complete.sort_values(
    "final_score",
    ascending=False
)[
    [
        "course_code",
        "course_title",
        "openai_aims_score",
        "openai_content_score",
        "openai_outcomes_score",
        "final_score"
    ]
].head(20)

,course_code,course_title,openai_aims_score,openai_content_score,openai_outcomes_score,final_score
5,EBA 3630,Data Driven Management Accounting,0.575792,0.573533,0.488051,0.553066
315,BUST08032,Business Analytics and Information Systems,0.514690,0.632385,0.483854,0.548174
52,GRA 2269,Leading in Organizations Using Intelligent Dec...,0.566667,0.494310,0.512697,0.527850
9,EDI 3510,Business Information Systems,0.566226,0.508026,0.422313,0.509878
115,TEM 0056,Applied Data Analytics,0.653757,0.383777,0.430964,0.503566
58,GRA 4137,Data Protection and Ethics in the Modern Busin...,0.568707,0.463886,0.451255,0.502657
365,BUST10145,Predictive Analytics for Business,0.478578,0.472222,0.567909,0.498686
798,ADM-12350,Data-driven business decisions,0.622188,0.412467,0.416186,0.497285
373,BUST10159,The Strategy of Digital Transformation,0.504358,0.529203,0.439358,0.496804
96,TEM 0241,AI for Business,0.474669,0.512872,0.501811,0.494825
